# Bonus: How to trace and evaluate RAG with MLflow

**CPU | OpenAI API key | Bonus**

## Goal

Use MLflow to inspect an OpenAI call, follow a RAG request through retrieval and generation, and score the answers
from those same traces. We will compare a missing-evidence request with its repaired version, then read what the
scores tell us about each answer.

## What you will learn

- Find messages, responses, timings and token usage in an automatically recorded OpenAI trace.
- Add spans for the Python operations between model calls, including retrieved and packed evidence.
- Compare token F1 with an LLM judge on a correct answer, a paraphrase and a wrong duration.
- Score saved RAG traces and inspect the evidence and rationale behind each verdict.

We keep the system we already understand and give its observations one place to live:

| What we built | What we connect here |
|---|---|
| Lab 01: vectors and ranked search | The same NumPy vector store, with retrieved policy IDs, text and scores visible in a span |
| Lab 02: evidence packing and cited answers | The exact packed evidence, messages, answer and claims inside the request trace |
| Lab 04: a trace record per question | A connected tree of timed spans that we can open and inspect |
| Lab 06: reference records and metrics | Code scores and expected answers in an evaluation run |
| Lab 07: an LLM judge checked by a person | A built-in judge's verdict and rationale, still open to human review |

The search stays the same so we can follow familiar evidence through the tool. MLflow stores and connects the
observations; the policies, answering rules and meaning of each score remain ours.


## Set up the notebook

Run Lab 00 first. Follow this lab's [README](README.md) to install the optional MLflow packages and start its local
server before running these cells. The server stores traces and evaluations; the notebook sends records to it.
On Colab, Setup installs the packages and starts the server inside the runtime.


In [1]:
# Setup: find the repository; on Colab, install the bonus and start its local server.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r Bonus_MLflow_Tracing_And_Evaluation/requirements.txt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())
sys.path.insert(0, str(ROOT))
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
print("Settings file found:", (ROOT / "models.env").name)


Settings file found: models.env


In [2]:
# Colab only: a browser on your laptop cannot use the runtime's localhost directly.
if IN_COLAB:
    import subprocess
    import time
    from urllib.request import urlopen
    from urllib.error import URLError

    server_folder = ROOT / ".cache" / "mlflow-bonus"
    server_folder.mkdir(parents=True, exist_ok=True)
    server_log = open(server_folder / "server.log", "a")
    server = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", "5057",
         "--backend-store-uri", "sqlite:///.cache/mlflow-bonus/mlflow.db",
         "--artifacts-destination", ".cache/mlflow-bonus/artifacts", "--workers", "1"],
        cwd=ROOT, stdout=server_log, stderr=server_log)
    for attempt in range(30):
        try:
            urlopen("http://127.0.0.1:5057/health", timeout=1).close()
            break
        except URLError:
            time.sleep(1)
    print("Server ready:", urlopen("http://127.0.0.1:5057/health").status == 200)


Config reads the shared settings and key, then connects both tracing and evaluation to the same experiment.
The two endpoint variables keep MLflow's separate judge client on the endpoint selected for these labs.


In [3]:
# Config: shared model settings and a key; MLflow's judges use the same endpoint.
import mlflow
from shared.settings import load_settings, load_key, stamp, chat_url, chat_client, chat_options

settings = load_settings()
assert load_key("OPENAI_API_KEY"), "Add OPENAI_API_KEY as explained in Lab 00."
os.environ["OPENAI_BASE_URL"] = chat_url(settings)
os.environ["OPENAI_API_BASE"] = chat_url(settings)
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5057"))
experiment = mlflow.set_experiment("RAG MLflow bonus")
mlflow.openai.autolog()
print("MLflow:", mlflow.__version__)
print("Tracking server:", mlflow.get_tracking_uri())
print(stamp(settings))


MLflow: 3.17.2
Tracking server: http://127.0.0.1:5057
{'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-28'}


## Part A: Inspect an OpenAI call in MLflow

MLflow began with tracking machine-learning experiments and models. It also records LLM traces, runs evaluations
and manages prompts. In this lab we use one **experiment**, a container for related work. A **trace** records one
request; a **span** records an operation within it. An evaluation **run** groups scores from one evaluation.

`mlflow.openai.autolog()` instruments the OpenAI SDK. After enabling it, we make the same call we normally would.
MLflow records the call's input messages, response, model, timing, token usage and any exception. It does not
need us to log each of those fields ourselves.

Start with a question and the evidence in the message. The model receives this text; MLflow records it as well.


In [4]:
messages = [{"role": "user", "content": (
    "Policy: Atlas reset links expire after 30 minutes.\n"
    "Question: How long is an Atlas reset link valid? Answer briefly.") }]
client = chat_client(settings)
reply = client.chat.completions.create(**chat_options("CHAT_MODEL", settings), messages=messages)
print(reply.choices[0].message.content)
print("Tokens:", reply.usage.prompt_tokens, "input;", reply.usage.completion_tokens, "output")


An Atlas reset link is valid for 30 minutes.
Tokens: 32 input; 11 output


Now retrieve the trace that autologging just created. A trace ID identifies this request independently of its
answer text. Flushing waits for pending trace writes before reading the stored object.


In [5]:
mlflow.flush_trace_async_logging()
openai_id = mlflow.get_last_active_trace_id()
openai_trace = mlflow.get_trace(openai_id)
print("Trace ID:", openai_id)
for span in openai_trace.data.spans:
    print(span.name, "|", span.span_type, "|", span.status.status_code)
    print("Input:", span.inputs)
    print("Response:", span.outputs["choices"][0]["message"]["content"])
    print("Usage:", span.outputs["usage"])


Trace ID: tr-f256267742182767dcfff2571f842008
Completions | CHAT_MODEL | SpanStatusCode.OK
Input: {'model': 'gpt-4o-mini', 'temperature': 0, 'messages': [{'role': 'user', 'content': 'Policy: Atlas reset links expire after 30 minutes.\nQuestion: How long is an Atlas reset link valid? Answer briefly.'}]}
Response: An Atlas reset link is valid for 30 minutes.
Usage: {'completion_tokens': 11, 'prompt_tokens': 32, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}


Trace(trace_id='tr-f256267742182767dcfff2571f842008')

In [7]:
# On Colab use its port proxy; elsewhere open the server URL or the forwarded Codespaces port.
from IPython.display import display, Markdown

if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_in_window(5057)
else:
    display(Markdown(f"[Open the MLflow UI]({mlflow.get_tracking_uri()})"))


[Open the MLflow UI](http://127.0.0.1:5057)

Open **RAG MLflow bonus**, choose **Traces**, then open the row with the printed ID. Inspect the messages, response
and token usage. The span's duration measures elapsed time for that call; token counts measure the text processed.
An `OK` status means the call completed successfully. It does not judge the answer.

**Checkpoint A:** find the same answer and input/output token counts in Python and the UI. The API request was
captured automatically. The next part adds operations that happen outside the SDK.


## Part B: Follow retrieval, packing and generation in one trace

A RAG request has work between model calls: ranking chunks, choosing evidence and assembling messages. OpenAI
tracing alone cannot name those Python stages. We will record them as child spans under one RAG request.

Use only our five Atlas/Boreal policies so every retrieved record can be read. This is the same search and
400-token packing logic used in Lab 04; we are adding instrumentation around it.


In [6]:
import json
from shared.search import chunk_vectors, search
from shared.answer import RULES, pack, parse, response_format

policies = json.loads((ROOT / "data" / "support-case.json").read_text())
corpus = [{**p, "chunk_id": p["source_id"]} for p in policies]
display(corpus[1])
vectors = chunk_vectors(corpus, settings)
print("Policy vectors:", vectors.shape)


{'source_id': 'H02',
 'title': 'Atlas expiry',
 'text': 'Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.',
 'version': '2026-10-01',
 'access': 'public',
 'chunk_id': 'H02'}

Policy vectors: (5, 1536)


Each row of `vectors` represents one policy, in `corpus` order. H02 is the Atlas expiry policy: it supplies the
30-minute answer. `chunk_vectors()` embeds these policies once and caches the vectors; it does not re-embed them
inside every question's trace.

First inspect a retrieval result. `search()` returns `(score, chunk)` pairs, best first. We then represent one
result as an MLflow `Document`: `id` identifies the chunk, `page_content` holds its text, and `metadata` holds the
ranking score. This representation lets MLflow's RAG judges read the retrieved evidence later.


In [7]:
from mlflow.entities import Document

question = "When does the password recovery email stop working for Atlas?"
hits = search(question, corpus, vectors, settings, k=3)
score, chunk = hits[0]
document = Document(id=chunk["chunk_id"], page_content=chunk["text"], metadata={"score": score})
display(document)
print("Ranking:", [(c["chunk_id"], round(s, 4)) for s, c in hits])


Document(page_content='Atlas passwords can be reset by email. Temporary accounts require a support agent.', metadata={'score': 0.5889610648617918}, id='H01')

Ranking: [('H01', 0.589), ('H04', 0.4685), ('H02', 0.4312)]


Next, record just the packing operation. A span is a timed `with` block. `set_inputs()` records the data it
received; `set_outputs()` records what it produced. The example is a separate, small trace named `pack-demo`.
In the complete function below, the same block runs inside a RAG request and becomes its child.


In [8]:
with mlflow.start_span(name="pack-demo") as span:
    span.set_inputs({"retrieved_ids": [c["chunk_id"] for _, c in hits], "budget": 400})
    lines = pack(hits, settings, budget=400)
    span.set_outputs({"evidence": lines})
print("Packed evidence:")
print("\n".join(lines))


Packed evidence:
[H01] Atlas passwords can be reset by email. Temporary accounts require a support agent.
[H04] Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.


Trace(trace_id='tr-745583268decf65b877ec196fad34e6a')

We can now join these operations into one function. `@mlflow.trace` creates the outer request span and records
its question, k and returned answer. Every `with` block inside it becomes a child span. The OpenAI calls made
inside those blocks become children automatically, including the query embedding during search.

The retrieval span records `Document` objects. Packing records the selected evidence and actual messages sent to
the answer model, plus the prompt fingerprint from Lab 04. Generation keeps the parsed answer and claims; its
SDK child keeps the raw reply and token usage. We put the model/endpoint stamp in trace metadata.


In [9]:
import hashlib

@mlflow.trace(name="policy-rag", span_type="CHAIN")
def rag(question: str, k: int = 3) -> str:
    mlflow.update_current_trace(metadata={"settings": json.dumps(stamp(settings))})
    with mlflow.start_span(name="retrieve", span_type="RETRIEVER") as span:
        span.set_inputs({"query": question, "k": k})
        hits = search(question, corpus, vectors, settings, k=k)
        span.set_outputs([Document(id=c["chunk_id"], page_content=c["text"],
                                   metadata={"score": s}) for s, c in hits])
    with mlflow.start_span(name="pack") as span:
        span.set_inputs({"retrieved_ids": [c["chunk_id"] for _, c in hits], "budget": 400})
        lines = pack(hits, settings, budget=400)
        messages = [{"role": "system", "content": RULES},
                    {"role": "user", "content": f"Question: {question}\nEvidence:\n" + "\n".join(lines)}]
        span.set_outputs({"packed_ids": [line[1:line.index("]")] for line in lines],
                          "evidence": lines, "messages": messages,
                          "prompt_sha": hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:12]})
    with mlflow.start_span(name="generate") as span:
        span.set_inputs({"question": question})
        reply = client.chat.completions.create(**chat_options("CHAT_MODEL", settings),
            response_format=response_format(settings), messages=messages)
        parsed = parse(reply.choices[0].message.content)
        span.set_outputs(parsed)
    return parsed["answer"]


Trace(trace_id='tr-950df2e51ff6138bf4c02b21a286be59')

Run three requests: a direct question with k=3, then the paraphrase with k=1 and k=3. The policy remains unchanged;
only the paraphrase's retrieval depth changes. Predict whether H02 reaches each prompt before reading the answers.
Keep each trace ID immediately after its request, so later scoring selects these requests only.


In [10]:
requests = [("How long does an Atlas reset link remain valid?", 3), (question, 1), (question, 3)]
rag_ids = []
for q, k in requests:
    answer_text = rag(q, k)
    rag_ids.append(mlflow.get_last_active_trace_id())
    print("k =", k, "|", q, "|", answer_text)
mlflow.flush_trace_async_logging()
print(stamp(settings))


k = 3 | How long does an Atlas reset link remain valid? | Atlas reset links remain valid for 30 minutes.
k = 1 | When does the password recovery email stop working for Atlas? | Insufficient evidence.
k = 3 | When does the password recovery email stop working for Atlas? | The password recovery email for Atlas stops working after 30 minutes.
{'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-28'}


[Trace(trace_id='tr-c45f913a2deb2254c2edb9fdff39c3da'), Trace(trace_id='tr-e4d553db03db88b12b56e1e8df208859'), Trace(trace_id='tr-150e046f442853791354986519e71485')]

In [11]:
# Read the middle request, the paraphrase at k=1, directly from the stored spans.
example_trace = mlflow.get_trace(rag_ids[1])
for span in example_trace.data.spans:
    duration_ms = (span.end_time_ns - span.start_time_ns) / 1_000_000
    print(span.name, "|", span.span_type, "|", round(duration_ms, 1), "ms")
    if span.name in {"retrieve", "pack", "generate"}:
        print(span.outputs)


policy-rag | CHAIN | 976.6 ms
retrieve | RETRIEVER | 246.8 ms
[{'page_content': 'Atlas passwords can be reset by email. Temporary accounts require a support agent.', 'metadata': {'score': 0.5889610648617918}, 'id': 'H01'}]
Embeddings | EMBEDDING | 238.1 ms
pack | UNKNOWN | 0.1 ms
{'packed_ids': ['H01'], 'evidence': ['[H01] Atlas passwords can be reset by email. Temporary accounts require a support agent.'], 'messages': [{'role': 'system', 'content': 'Answer the question in one sentence, using only the evidence. List each claim the answer makes, with the ID of the evidence line it comes from and a quote: words copied exactly from that line. If the evidence does not answer the question, the answer is "Insufficient evidence." and the claims list is empty. Reply in JSON shaped like {"answer": "...", "claims": [{"claim": "...", "source_id": "...", "quote": "..."}]}.'}, {'role': 'user', 'content': 'Question: When does the password recovery email stop working for Atlas?\nEvidence:\n[H01] Atla

`UNKNOWN` is MLflow's default type for spans without an explicit type; the names `pack` and `generate` identify
our Python stages. Their inputs, outputs and timings are still recorded.

Open these three `policy-rag` traces in the UI. Expand the child spans and compare the `retrieve` and `pack`
outputs. In this small example the selected policies fit in 400 tokens. In a larger corpus, a retrieved chunk
can be absent from the pack: the span outputs keep that distinction visible.

The fields from Lab 04 now have a location: `hits` in retrieve, `packed` and `prompt_sha` in pack, the raw `reply`
and `usage` in the OpenAI generation span. The outer span connects them to one question.

**Checkpoint B:** locate H02 in the direct request and in the paraphrase at k=3, then inspect the k=1 request's
actual evidence and answer. A refusal when H02 is absent shows an evidence gap; it is not an exception or failed API
call. Reading one trace tells us what happened. Next we ask how to score an answer.


## Part C: Compare word overlap with an LLM judge

Consider three **constructed answers**, all to the Atlas duration question. The reference is `30 minutes`.
Before running a judge, decide which answers you would accept. These examples are not a calibration dataset.

This is Lab 06's reference-based scoring in MLflow's record format. An evaluation record keeps `inputs` (the question), `outputs` (the answer being scored), and `expectations`
(the reference). The reference is supplied to the evaluator; it was not given to an answering model here.


In [12]:
examples = [{"inputs": {"question": requests[0][0]}, "outputs": answer,
             "expectations": {"expected_response": "30 minutes"}}
            for answer in ["30 minutes", "half an hour", "60 minutes"]]
display(examples[1])


{'inputs': {'question': 'How long does an Atlas reset link remain valid?'},
 'outputs': 'half an hour',
 'expectations': {'expected_response': '30 minutes'}}

`Correctness` is a built-in LLM judge. We name its model explicitly using the configured `JUDGE_MODEL`; we also
pass the model's temperature/reasoning options. MLflow makes this separate model call. It reads the question,
answer and expected response and returns **Feedback**, including a verdict and a rationale.


In [13]:
from mlflow.genai.scorers import Correctness, RetrievalGroundedness

judge_options = chat_options("JUDGE_MODEL", settings)
judge_model = "openai:/" + judge_options.pop("model")
correctness = Correctness(model=judge_model, inference_params=judge_options)
feedback = correctness(**examples[1])
print("Verdict:", feedback.value)
print("Reason:", feedback.rationale)


Verdict: yes
Reason: The question asks how long an Atlas reset link remains valid, and the document clearly states 'half an hour,' which is equivalent to 30 minutes. Therefore, the claim that the reset link remains valid for 30 minutes is supported by the document.


Now bring back Lab 06's token F1. `@scorer` adapts a Python function to the same evaluation interface. The function
receives an output and its expectations, then returns a number. It makes no model call.

Against `30 minutes`, token F1 gives `half an hour` zero overlap, even though the durations agree. It gives the
wrong `60 minutes` half credit because the word *minutes* matches. A judge can assess the meaning; we must still
read its decisions and reasons.


In [14]:
from mlflow.genai import scorer
from shared.evals import em_f1

@scorer
def token_f1(outputs: str, expectations: dict) -> float:
    return em_f1(outputs, [expectations["expected_response"]])[1]

for example in examples:
    print(example["outputs"], "| token F1:", token_f1(
        outputs=example["outputs"], expectations=example["expectations"]))


30 minutes | token F1: 1.0
half an hour | token F1: 0.0
60 minutes | token F1: 0.5


`mlflow.genai.evaluate()` applies both scorers to the three records and logs an evaluation run. No `predict_fn`
is supplied, because the records already contain answers. The judge does make three new scoring calls, including
a second judgement of the paraphrase we just inspected.

The result table calls the stored answer `response`; the input record called it `outputs`. Each row also carries
its trace ID, so we can move from a score back to the recorded request.


In [15]:
example_results = mlflow.genai.evaluate(data=examples, scorers=[token_f1, correctness])
display(example_results.result_df[["response", "token_f1/value", "correctness/value", "correctness/rationale"]])
print(stamp(settings))


2026/10/28 19:53:10 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


,response,token_f1/value,correctness/value,correctness/rationale
0,30 minutes,1.0,yes,The document directly answers the question reg...
1,half an hour,0.0,yes,The claim states that an Atlas reset link rema...
2,60 minutes,0.5,no,The claim states that an Atlas reset link rema...


{'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-28'}


**Checkpoint C:** token F1 is 1.0, 0.0 and 0.5 in example order. Compare those numbers with your labels and the
judge's actual verdicts. Open **Evaluation runs** in the experiment and inspect a row's rationale. An aggregate
mean describes these three examples only; it is not a probability that a new answer will be correct.

The useful change is that code scores and judge assessments are stored together. Choosing a reference and checking
a judge's mistakes still belong to us. Next we apply that workflow to the RAG requests we already recorded.


## Part D: Score the RAG requests already traced

Our three Part B traces already hold questions, answers and retrieved evidence. Attach the expected duration to
each one, then evaluate exactly those trace objects. `log_expectation()` adds an annotation after generation;
it does not change the recorded prompt or answer.


In [16]:
import pandas as pd

for trace_id in rag_ids:
    mlflow.log_expectation(trace_id=trace_id, name="expected_response", value="30 minutes")
trace_data = pd.DataFrame({"trace": [mlflow.get_trace(trace_id) for trace_id in rag_ids]})
print("Saved RAG requests selected:", len(trace_data))


Saved RAG requests selected: 3


Use two criteria with different inputs:

| Scorer | Reads | Asks |
|---|---|---|
| Correctness | Question, answer, expected response | Does the answer supply the expected information? |
| RetrievalGroundedness | Question, answer, RETRIEVER evidence | Is the answer supported by the retrieved context? |

Groundedness reads the retriever span's documents. It does not automatically check only the evidence that packing
kept. Here the retrieved policies fit in the pack; on larger requests, inspect both stages. A supported refusal
can still fail correctness when the expected information is missing from the answer.


In [17]:
groundedness = RetrievalGroundedness(model=judge_model, inference_params=judge_options)
rag_results = mlflow.genai.evaluate(data=trace_data, scorers=[correctness, groundedness])
display(rag_results.result_df[["response", "correctness/value", "retrieval_groundedness/value"]])
print("Original request IDs:", rag_ids)
print(stamp(settings))


,response,correctness/value,retrieval_groundedness/value
0,Atlas reset links remain valid for 30 minutes.,yes,yes
1,Insufficient evidence.,no,no
2,The password recovery email for Atlas stops wo...,yes,yes


Original request IDs: ['tr-c45f913a2deb2254c2edb9fdff39c3da', 'tr-e4d553db03db88b12b56e1e8df208859', 'tr-150e046f442853791354986519e71485']
{'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-28'}


In [18]:
# Inspect the second request: the paraphrase at k=1.
row = rag_results.result_df.iloc[1]
for name in ["correctness", "retrieval_groundedness"]:
    print(name, ":", row[f"{name}/value"])
    print(row[f"{name}/rationale"], "\n")


correctness : no
The claim states that the password recovery email stops working for Atlas after 30 minutes. However, the document simply states 'Insufficient evidence' regarding when the password recovery email stops working. This indicates that the document does not provide any confirmation or details related to the claim, making it unsupported. 

retrieval_groundedness : no
The claim asks about when the password recovery email stops working for Atlas, but the document does not provide any information on the timing or conditions under which the password recovery email becomes ineffective. The document only states that passwords can be reset by email and mentions temporary accounts requiring a support agent, which does not address the claim. 



**Checkpoint D:** the evaluation contains three rows for the selected RAG requests. Open a row, read its
assessments and follow its trace to the retrieved evidence and actual prompt. In the saved run both judges reject the k=1 refusal. H01 says how to reset a password, but gives no expiry time.
The answer correctly reports that this evidence is insufficient. Read the groundedness rationale carefully:
a built-in judge's definition of a pass may differ from the behaviour we want. Here the groundedness rationale discusses the requested expiry fact instead of checking whether the refusal is
justified by H01. Its verdict needs the same inspection as the judge in Lab 07.

There is no `rag()` or `predict_fn` call in Part D: we reused the saved answers. The two judges still add model
calls. Re-scoring a trace saves generation work; it does not make evaluation free.

As in Lab 07, read a judge's reason against the actual evidence. These few examples show how to inspect a judge;
they do not establish agreement with people across the questions your application will receive.

**Lesson to Remember**

A trace records the path an answer took. Each scorer asks one named question about that answer or its evidence.
MLflow brings the records and scores together, so we can open a questionable score and inspect the request behind it.


## What you can do now

- Enable OpenAI tracing and inspect one recorded request.
- Add Python stages to a connected RAG trace, preserving retrieved and packed evidence separately.
- Compare a code metric with an LLM judge and inspect its rationale.
- Evaluate saved requests without generating their answers again.


## Challenge

Run the paraphrase with k=2. Before running it, predict whether H02 will enter the prompt. Record its trace ID,
attach the same expected response, and evaluate only that new trace with the two existing judges. Use its evidence,
answer and score rationales to explain whether the extra retrieved policy was enough.

Use `rag`, `get_last_active_trace_id`, `log_expectation` and the one-row `trace` DataFrame pattern from Part D.
This adds one answer and two judge evaluations. Keep your prediction even if your run disagrees.


In [21]:
# TODO: run the paraphrase with k=2, keep its trace ID, then score that trace.


## Troubleshooting

| Symptom | What to check |
|---|---|
| An empty 403 response in Config | Another service may own the port. On macOS, Control Center often owns 5000. Use the server on 5057 and match `MLFLOW_TRACKING_URI` to it. |
| Connection refused on port 5057 | Start the server from the README. Match its port to `MLFLOW_TRACKING_URI`, then restart the kernel if you changed the environment. |
| An empty Traces table | Choose **RAG MLflow bonus** and clear the time filter. Run `flush_trace_async_logging()` before reading a new trace. |
| A score is blank or contains an error | Inspect its error column in `result_df` and the configured judge model/key. A scorer error is not a failed quality check. |
| The judge uses a different endpoint | Config sets both OpenAI endpoint variables from `chat_url(settings)`. Re-run it after changing settings. |
| RAG judging cannot find context | Inspect the span type `RETRIEVER` and its Document outputs; a plain list of IDs does not supply the policy text. |
| Colab or Codespaces cannot open the UI | Use Colab's window link or the forwarded port URL, not your laptop's localhost. Python trace objects and score tables remain available. |

To run the notebook again, restart the kernel and run from Setup. Existing traces remain in the experiment; Part D
uses only the new IDs saved in that run. Stop the terminal server with Ctrl+C when finished. On Colab, the next
cell stops the server this notebook started.


In [22]:
if IN_COLAB:
    server.terminate()
    server_log.close()


## Going further

Try MLflow's retrieval relevance and sufficiency scorers when you need to score those separate questions. For judge
calibration, return to [Lab 07](../07_LLM_Judge_Calibration/README.md): record human labels first and inspect false
passes. A small demonstration of a tool does not validate its judge for your application.

## Resources

- [MLflow: OpenAI tracing](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/openai/)
- [MLflow: manual tracing](https://mlflow.org/docs/latest/genai/tracing/app-instrumentation/manual-tracing/)
- [MLflow: GenAI evaluation](https://mlflow.org/docs/latest/genai/eval-monitor/quickstart/)
- [MLflow: RAG judges](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/rag/)
- [MLflow: evaluate existing traces](https://mlflow.org/docs/latest/genai/eval-monitor/running-evaluation/traces/)

Back to [the lab list](../README.md).
